# Data cleaning - NTSB aviation cases

## Setup and loading

In [1]:
import json

import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

pd.set_option("display.max_colwidth", 200)

In [2]:
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

True

In [3]:
with open("ntsb_cases.json", encoding="utf-8") as file:
    raw_cases = json.load(file)

In [4]:
print("Number of cases:", len(raw_cases))

Number of cases: 8270


## Build the table

In [5]:
def clean_text(value):
    if isinstance(value, str) and value.strip() != "":
        return value.strip()
    return None

In [6]:
case_rows = []
for case in raw_cases:
    narratives = case.get("narratives") or []
    if len(narratives) > 0:
        narrative = narratives[0]
    else:
        narrative = {}

    aircraft_list = case.get("aircrafts") or []
    if len(aircraft_list) > 0:
        aircraft_category = aircraft_list[0].get("aircraftCategory")
    else:
        aircraft_category = None

    defining_events = []
    for aircraft in aircraft_list:
        for event in aircraft.get("events") or []:
            if event.get("isDefiningEvent"):
                defining_events.append(event)
    if len(defining_events) > 0:
        defining_category = defining_events[0].get("cicttEventSOEGroup")
    else:
        defining_category = None

    case_rows.append({
        "mKey": case.get("mKey"),
        "ntsbNumber": case.get("ntsbNumber"),
        "eventDate": case.get("eventDate"),
        "eventType": case.get("eventType"),
        "aircraftCategory": aircraft_category,
        "analysisNarrative": clean_text(narrative.get("analysisNarrative")),
        "numberOfDefiningEvents": len(defining_events),
        "definingCategory": defining_category,
    })

In [7]:
cases_table = pd.DataFrame(case_rows)

In [8]:
cases_table.shape

(8270, 8)

In [9]:
cases_table.head()

,mKey,ntsbNumber,eventDate,eventType,aircraftCategory,analysisNarrative,numberOfDefiningEvents,definingCategory
0,102484,WPR21LA077,2021-01-01,Accident,Helicopter,"The helicopter pilot reported that, while hover taxing on the taxiway, a Cessna airplane taxied from the ramp onto the same taxiway and turned toward and approached the helicopter. The helicopter ...",1,Loss Of Control (Inflight)
1,102492,WPR21LA079,2021-01-01,Accident,Airplane,"The pilot reported that, following a “smooth” landing, he reduced power, but the throttle control broke and the engine advanced to full power. The pilot immediately placed the mixture control in t...",1,System/Component Failure (Powerplant)
2,102503,WPR21WA081,2021-01-01,Accident,Helicopter,NaN,1,Loss Of Control (Inflight)
3,102480,CEN21LA104,2021-01-02,Accident,Airplane,"As the airplane approached the destination, the pilot requested VFR (visual flight rules) flight following and was issued a discrete beacon code. He asked if there were any pilot reports (PIREPs) ...",1,Loss Of Control (Inflight)
4,102485,WPR21LA078,2021-01-02,Accident,Airplane,The pilot was departing on the first flight of the day in cold weather conditions after the airplane had spent several hours inside a hangar to melt ice from the airframe and warm the engine. Abou...,1,System/Component Failure (Powerplant)


## Cases with text

In [10]:
cases_table = cases_table[cases_table["analysisNarrative"].notna()]

In [11]:
print("Cases with analysisNarrative:", len(cases_table))

Cases with analysisNarrative: 5580


## Cases with one defining event

In [12]:
cases_table = cases_table[cases_table["numberOfDefiningEvents"] == 1]
cases_table = cases_table.rename(columns={"definingCategory": "label"})

In [13]:
print("Cases with one defining event:", len(cases_table))

Cases with one defining event: 5501


## Drop uninformative labels

In [14]:
UNINFORMATIVE_LABELS = ["Unknown", "Other"]
cases_table = cases_table[~cases_table["label"].isin(UNINFORMATIVE_LABELS)]

In [15]:
print("Cases after dropping Unknown and Other:", len(cases_table))

Cases after dropping Unknown and Other: 5056


## Drop rare classes
Rare class is a class with <30 cases

In [16]:
MINIMUM_CASES_PER_CLASS = 30
label_counts = cases_table["label"].value_counts()
frequent_labels = label_counts[label_counts >= MINIMUM_CASES_PER_CLASS].index
cases_table = cases_table[cases_table["label"].isin(frequent_labels)]

In [17]:
print("Cases:", len(cases_table))
print("Classes:", cases_table["label"].nunique())

Cases: 4861
Classes: 17


In [18]:
cases_table["label"].value_counts()

label
Loss Of Control (Ground)                     924
Loss Of Control (Inflight)                   806
Abnormal Runway Contact                      752
System/Component Failure (Powerplant)        562
Fuel                                         517
System/Component Failure (Non-Powerplant)    314
Collision During Takeoff/Land                183
Low Altitude Operation/Event                 155
Runway Excursion                             125
CFIT                                          85
Turbulence                                    85
Unintended Flight Into IMC                    81
Birdstrike                                    71
Undershoot/Overshoot                          69
Ground Collision                              53
Fire                                          46
Ground Handling                               33
Name: count, dtype: int64

## Processed text for TF-IDF

In [ ]:
STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [20]:
def tokenize(text):
    text = text.lower().replace("’", "'").replace("-", " ").replace("/", " ")
    words = []
    for token in word_tokenize(text):
        if token == "n't":
            token = "not"
        if token.isalpha() and len(token) >= 2 and token not in STOPWORDS:
            words.append(token)
    return words

In [21]:
cases_table["tokens"] = cases_table["analysisNarrative"].apply(tokenize)

In [22]:
def lemmatize_words(words):
    lemmas = []
    for word in words:
        lemma = lemmatizer.lemmatize(word)
        if len(lemma) >= 2:
            lemmas.append(lemma)
    return " ".join(lemmas)

In [23]:
cases_table["text_lemmatized"] = cases_table["tokens"].apply(lemmatize_words)

In [24]:
for _, row in cases_table.sample(3, random_state=1).iterrows():
    print("ORIGINAL  :", row["analysisNarrative"][:300])
    print("LEMMATIZED:", row["text_lemmatized"][:300])
    print()

ORIGINAL  : The pilot reported that, while the helicopter was in the hanger, he put 4 gallons of fuel into the left fuel tank. He then intended to hover taxi to the fuel island for fuel. He observed people and an aircraft near the fuel island, and he hover taxied to a dirt area. While about 12-15 ft above groun
LEMMATIZED: pilot reported helicopter hanger put gallon fuel left fuel tank intended hover taxi fuel island fuel observed people aircraft near fuel island hover taxied dirt area ft ground level engine complete loss power reported low autorotate landed hard uneven terrain collapsed left skid inspection revealed 

ORIGINAL  : The pilot was performing a touch and go landing in a tailwheel-equipped airplane. As he added power after the landing, he experienced what he assumed was a gust of wind from the right side. The airplane’s right wing lifted, and it veered left into the trees impacting several. The airplane sustained 
LEMMATIZED: pilot performing touch go landing tailwheel equi

## Final checks and save

In [25]:
cases_table.shape

(4861, 10)

In [26]:
cases_table[["analysisNarrative", "text_lemmatized", "label"]].isna().sum()

analysisNarrative    0
text_lemmatized      0
label                0
dtype: int64

In [28]:
OUTPUT_COLUMNS = ["mKey", "ntsbNumber", "eventDate", "eventType", "aircraftCategory",
                  "label", "analysisNarrative", "text_lemmatized"]
cases_table[OUTPUT_COLUMNS].to_json("clean_cases.json", orient="records", force_ascii=False, indent=1)

In [29]:
print("Saved", len(cases_table), "cases to clean_cases.json")

Saved 4861 cases to clean_cases.json
